# Predicting student dropout and academic success

ML Assignment 2: exploratory analysis and a walkthrough of the training in `train.py`.

**Task.** Predict each student's outcome at the end of the normal course duration:
**Dropout**, **Enrolled** (still studying) or **Graduate**. This is three-class classification.

**Prediction point.** The features include first- and second-semester results, so the
prediction is made at the end of the first academic year, not at admission.

This notebook imports the same code as `train.py` (`data_loader`, `metrics` and the model
definitions), so its numbers match the saved models. It trains in memory only and does
**not** overwrite `model/`.

In [ ]:
import os
import sys
from pathlib import Path

# Work from the repository root so imports and data/raw/ resolve the same way as train.py.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from data_loader import CLASSES, FEATURE_COLS, NOMINAL_COLS, NUMERIC_COLS, TARGET, load_raw
from metrics import METRIC_NAMES, classification_report_frame, compute_metrics, confusion_matrix_figure
from train import MODELS, SEED, build_pipeline, build_preprocessor, split_data

%matplotlib inline

## 1. Load the data

In [ ]:
df = load_raw()   # downloads the UCI zip into data/raw/ on first run
print(df.shape)
df.head()

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Nominal (one-hot) columns:", len(NOMINAL_COLS), "| numeric columns:", len(NUMERIC_COLS))
df[NOMINAL_COLS].nunique().sort_values(ascending=False)

## 2. Exploratory analysis

### Class balance
Graduate is about half the data and Enrolled under a fifth. This is why the models are
compared on **macro** F1, where each class counts equally.

In [ ]:
counts = df[TARGET].value_counts().reindex(CLASSES)
display(pd.DataFrame({"students": counts, "share": (counts / len(df)).round(3)}))
ax = counts.plot.bar(rot=0, color=["#c0504d", "#f2a541", "#4f81bd"], title="Students per outcome")
ax.set_ylabel("students");

### First-year progress separates the classes
Units *approved* in the second semester, by outcome: dropouts approve far fewer units, and
Enrolled students sit in between. This in-between position makes Enrolled the hardest class.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.boxplot(data=df, x=TARGET, y="Curricular units 2nd sem (approved)", order=CLASSES, ax=ax)
ax.set_title("2nd-semester units approved by outcome");

In [ ]:
# Financial status: share of each outcome among students whose tuition fees are / are not up to date
pd.crosstab(df["Tuition fees up to date"], df[TARGET], normalize="index").round(2)

### Correlated features
First- and second-semester features are strongly correlated. This matters for Gaussian
Naive Bayes, which assumes features are independent given the class.

In [ ]:
sem = [c for c in NUMERIC_COLS if c.startswith("Curricular units")]
corr = df[sem].corr()
pairs = {c.replace("Curricular units 1st sem ", ""): round(corr.loc[c, c.replace("1st", "2nd")], 2)
         for c in sem if "1st" in c}
pd.Series(pairs, name="corr(1st sem, 2nd sem)")

## 3. Train/test split and preprocessing

- A stratified 80/20 split (`random_state=42`). The 20% hold-out is what `test_data.csv`
  contains, and it is not used for tuning.
- The nine nominal codes are **one-hot encoded**; the other 27 columns are **standardised**.
- The preprocessing sits inside each pipeline, so a saved model takes the raw 36 columns directly.

In [ ]:
X_train, X_test, y_train, y_test = split_data(df)
print(len(X_train), "training rows,", len(X_test), "hold-out rows")
n_inputs = build_preprocessor().fit(X_train).transform(X_train.head(1)).shape[1]
print("Inputs to each model after preprocessing:", n_inputs)

## 4. Tune and train the five models

The same settings as `train.py`: 5-fold stratified CV on the training split, scored by
macro F1, and class weighting where the estimator supports it (about 30-60 seconds).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fitted, results = {}, {}
for name, (_, estimator, grid) in MODELS.items():
    search = GridSearchCV(build_pipeline(estimator), grid, scoring="f1_macro", cv=cv, n_jobs=-1)
    search.fit(X_train, y_train)
    model = fitted[name] = search.best_estimator_
    results[name] = compute_metrics(y_test, model.predict(X_test), model.predict_proba(X_test),
                                    [str(c) for c in model.classes_])
    print(f"{name:24s} best {search.best_params_}  CV macro F1 = {search.best_score_:.4f}")

## 5. Hold-out metrics (primary criterion: macro F1)

In [ ]:
table = pd.DataFrame(results).T[METRIC_NAMES].astype(float).round(4)
display(table.sort_values("F1", ascending=False))

# Same numbers as the saved models report?
saved = pd.DataFrame(json.loads(Path("model/metrics.json").read_text())).T[METRIC_NAMES].astype(float).round(4)
print("Matches model/metrics.json:", table.equals(saved.loc[table.index]))

## 6. Confusion matrices

In [ ]:
for name, model in fitted.items():
    fig = confusion_matrix_figure(y_test, model.predict(X_test), CLASSES, title=name)
    plt.show()

In [ ]:
# Per-class view: Enrolled is the weakest class for every model
pd.DataFrame({name: classification_report_frame(y_test, m.predict(X_test), CLASSES)["f1-score"]
              for name, m in fitted.items()}).loc[CLASSES].round(3)

## 7. Observations

- **Logistic Regression** and **SVM** tie on macro F1 (about 0.70). Logistic Regression has
  the better AUC and is far simpler, so it is the preferred model.
- **KNN** looks reasonable on accuracy but poor on macro F1. Without class weighting it rarely
  predicts Enrolled.
- **Decision Tree** over-predicts Enrolled, which costs Graduate recall.
- **Gaussian Naive Bayes** is weakest: correlated semester features and one-hot 0/1 columns
  break its independence and Gaussian assumptions.
- Every model finds **Enrolled** the hardest class.

The README has the full observations table with the supporting counts.